<a href="https://colab.research.google.com/github/RMoulla/IA_BRGM/blob/main/Sentiment_Analysis_Recommandation_Correction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Analyse du sentiment avec le machine learning
Dans ce TP, nous nous intéressons à un problème de classification de textes en utilisant des techniques de machine learning, plus spécifiquement à une tâche d'analyse de sentiments (sentiment analysis).

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score
import re

## Chargement des données.

Les données sont des avis de jeux vidéo. La colonne `text` contient le commentaire et `recommended` indique si l'auteur recommande le jeu.
Les étudiants sont encouragés à lire quelques commentaires pour se faire une idée des données manipulées.

Dans Colab, importez le fichier `Reviews.csv` du dépôt dans les fichiers de la session avant d'exécuter la cellule suivante. Conservez son nom exact.


In [ ]:
reviews = pd.read_csv('Reviews.csv')
reviews.head(20)

Transformer les labels en entiers.

In [ ]:
reviews['recommended'] = reviews['recommended'].astype(dtype=np.int64)
reviews.head()

## Pré-traitement des données


In [ ]:
# Fonction qui supprime des caractères spéciaux pour nettoyer les commentaires et réduire le bruit.
def preprocess(text):

    text = text.lower()
    text=re.sub('(\\d|\\W)+',' ', text)
    return text


In [ ]:
text = 'Ceci est un Exemple qui contient les chiffres 124 et les caractères !$'
preprocess(text)

In [ ]:
reviews['text'] = reviews['text'].apply(lambda x:preprocess(x))
reviews.head(15)

In [ ]:
# Visualisation de la variable d'intérêt
reviews['recommended'].value_counts().plot(kind='bar')


## Séparation des données d'entraînement et de test

On sépare les textes et leurs labels avant d'apprendre le vocabulaire. Le découpage est reproductible et stratifié : il conserve la proportion d'avis recommandés dans les deux ensembles.

Le jeu de test est réservé à l'évaluation finale. Le choix de l'hyperparamètre `C` reposera uniquement sur une validation croisée des données d'entraînement.


In [ ]:
from sklearn.model_selection import train_test_split

text_train, text_test, y_train, y_test = train_test_split(
    reviews['text'], reviews['recommended'],
    train_size=0.75, random_state=42, stratify=reviews['recommended']
)


## Vectorisation

La vectorisation transforme chaque commentaire en un vecteur contenant le nombre d'occurrences de chaque mot. Le vocabulaire est appris uniquement sur les textes d'entraînement.

Cette première vectorisation permet d'examiner les mots et les vecteurs. Lors de la validation croisée, le pipeline réapprendra le vocabulaire dans chaque pli d'entraînement afin de ne pas utiliser les textes du pli de validation.


In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer()
X_train = vectorizer.fit_transform(text_train)


In [ ]:
print(vectorizer.get_feature_names_out())

In [ ]:
with np.printoptions(threshold=np.inf):
    print(X_train.toarray()[0])


## Modèle de régression logistique

On implémente une régression logistique pour obtenir une performance de référence. Son hyperparamètre de régularisation `C` est sélectionné par validation croisée sur les seuls textes d'entraînement.

Le pipeline regroupe la vectorisation, la mise à l'échelle et le modèle. Ainsi, chaque pli de validation est évalué avec un vocabulaire et une mise à l'échelle appris sans ses propres textes. `with_mean=False` conserve la représentation creuse des données.


In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, StratifiedKFold

pipeline = Pipeline([
    ('vectorizer', CountVectorizer()),
    ('scaler', StandardScaler(with_mean=False)),
    ('classifier', LogisticRegression(max_iter=1000, random_state=42)),
])
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
search = GridSearchCV(
    pipeline,
    param_grid={'classifier__C': [0.5, 1, 1.5, 2, 2.5, 3]},
    scoring='accuracy', cv=cv, refit=True, n_jobs=1
)
search.fit(text_train, y_train)

print(f"C retenu : {search.best_params_['classifier__C']}")
print(f"Accuracy moyenne en validation croisée : {search.best_score_:.3f}")


Le meilleur pipeline a été réentraîné automatiquement sur tout le jeu d'entraînement. On récupère ses étapes pour interpréter le modèle et appliquer exactement la même transformation aux données de test.


In [ ]:
best_pipeline = search.best_estimator_
vectorizer = best_pipeline.named_steps['vectorizer']
scaler = best_pipeline.named_steps['scaler']
final_model = best_pipeline.named_steps['classifier']

X_train = vectorizer.transform(text_train)
X_test = vectorizer.transform(text_test)
X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)

print('Accuracy finale sur le test : %.3f' % accuracy_score(
    y_test, final_model.predict(X_test_scaled)
))


## Analyse des features du modèle


Nous allons maintenant examiner les coefficients de la régression logistique associés aux différents features (tokens).

In [ ]:
# Extraire la liste des features
feature_list = vectorizer.get_feature_names_out()

# Extraire la liste des coefficients
coef_list = final_model.coef_[0]

# Associer les features aux coefficients avec un format dictionnaire
feature_to_coef = dict(zip(feature_list, coef_list))

In [ ]:
print('Positive Words')
for best_positive in sorted(
    feature_to_coef.items(),
    key=lambda x: x[1],
    reverse=True)[:10]:
    print(best_positive)

In [ ]:
print('Negative Words')
for best_negative in sorted(
    feature_to_coef.items(),
    key=lambda x: x[1],
    reverse = False)[:10]:
    print(best_negative)

## Modèle random forest

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(random_state=42)
rf.fit(X_train_scaled, y_train)

y_pred = rf.predict(X_test_scaled)
print(f'Accuracy: {accuracy_score(y_test, y_pred)}')

## Modèle Naive Bayes

In [ ]:
from sklearn.naive_bayes import MultinomialNB

nb = MultinomialNB()
nb.fit(X_train_scaled, y_train)

y_pred = nb.predict(X_test_scaled)
print(f'Accuracy: {accuracy_score(y_test, y_pred)}')


## Comparaison des différents modèles

Les performances des trois modèles (régression logistique, random forest et Naive Bayes) sont comparées sur le jeu de test à l'aide d'une courbe ROC. Tous reçoivent les données transformées avec le vocabulaire et la mise à l'échelle appris sur le jeu d'entraînement.


In [ ]:
from sklearn.metrics import roc_curve, auc
import matplotlib.pyplot as plt

prob_lr = final_model.predict_proba(X_test_scaled)[:, 1]  # Pour le modèle logit
prob_rf = rf.predict_proba(X_test_scaled)[:, 1]  # Pour le modèle random forest
prob_nb = nb.predict_proba(X_test_scaled)[:, 1]  # Pour le modèle Naive Bayes

fpr_lr, tpr_lr, _ = roc_curve(y_test, prob_lr)
roc_auc_lr = auc(fpr_lr, tpr_lr)

fpr_rf, tpr_rf, _ = roc_curve(y_test, prob_rf)
roc_auc_rf = auc(fpr_rf, tpr_rf)

fpr_nb, tpr_nb, _ = roc_curve(y_test, prob_nb)
roc_auc_nb = auc(fpr_nb, tpr_nb)

plt.figure()
plt.plot(fpr_lr, tpr_lr, color='blue', lw=2, label='Modèle logit (AUC = %0.2f)' % roc_auc_lr)
plt.plot(fpr_rf, tpr_rf, color='green', lw=2, label='Modèle random forest (AUC = %0.2f)' % roc_auc_rf)
plt.plot(fpr_nb, tpr_nb, color='orange', lw=2, label='Modèle Naive Bayes (AUC = %0.2f)' % roc_auc_nb)

plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('Taux de faux positifs')
plt.ylabel('Taux de vrais positifs')
plt.title('Comparaison des courbes ROC')
plt.legend(loc="lower right")
plt.show()
